In [10]:
#stt 서비스

In [11]:
import os
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk
import json
from openai import AzureOpenAI

load_dotenv()

True

In [12]:
speech_key = os.getenv("AZURE_SPEECH_KEY")
speech_endpoint = os.getenv("AZURE_SPEECH_ENDPOINT")


openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
openai_subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

service_region = os.getenv("AZURE_SPEECH_REGION")

In [13]:
#stt 서비스
def recognize_from_microphone():
 
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, endpoint=speech_endpoint)
    speech_config.speech_recognition_language="ko-kr"

    audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
    speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

    print("Speak into your microphone.")
    speech_recognition_result = speech_recognizer.recognize_once_async().get()

    if speech_recognition_result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print("Recognized: {}".format(speech_recognition_result.text))
    elif speech_recognition_result.reason == speechsdk.ResultReason.NoMatch:
        print("No speech could be recognized: {}".format(speech_recognition_result.no_match_details))
    elif speech_recognition_result.reason == speechsdk.ResultReason.Canceled:
        cancellation_details = speech_recognition_result.cancellation_details
        print("Speech Recognition canceled: {}".format(cancellation_details.reason))
        if cancellation_details.reason == speechsdk.CancellationReason.Error:
            print("Error details: {}".format(cancellation_details.error_details))
            print("Did you set the speech resource key and endpoint values?")

recognize_from_microphone()

Speak into your microphone.
No speech could be recognized: NoMatchDetails(reason=NoMatchReason.InitialSilenceTimeout)


In [16]:
#만약 OpenAI STT + TTS
def stt_openai_tts(
    endpoint,
    subscription_key,
    deployment,
    speech_key,
    service_region,
    voice_name="ko-KR-HyunsuMultilingualNeural"
):
    # 1. STT: 마이크로부터 음성 인식
    print("마이크에 대고 말씀해주세요...")
    
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    speech_config.speech_recognition_language = "ko-KR"
    
    audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
    speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)
    
    speech_recognition_result = speech_recognizer.recognize_once_async().get()
    user_question = speech_recognition_result.text
    print(f"인식된 텍스트: {user_question}")
    
    # 2. OpenAI로 답변 생성
    print("OpenAI에게 질문 중...")
    
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=subscription_key,
        api_version="2025-01-01-preview",
    )
    
    completion = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": "사용자가 정보를 찾는 데 도움이 되는 AI 도우미입니다."},
            {"role": "user", "content": user_question}
        ],
        max_tokens=800,
        temperature=0.7
    )
    
    response_text = completion.choices[0].message.content
    print(f"OpenAI 응답: {response_text}")
    
    # 3. TTS: 텍스트를 음성으로 출력
    print("음성으로 답변 중...")
    
    tts_speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    tts_speech_config.speech_synthesis_voice_name = voice_name
    
    speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=tts_speech_config)
    speech_synthesizer.speak_text_async(response_text).get()
    
    print("완료!")
    return {"recognized_text": user_question, "ai_response": response_text}

# 사용 예시

if __name__ == "__main__":
    result = stt_openai_tts(
        endpoint=openai_endpoint,
        subscription_key=openai_subscription_key,
        deployment=deployment,
        speech_key=speech_key,
        service_region=service_region
    )
    
    print(json.dumps(result, indent=2, ensure_ascii=False))

마이크에 대고 말씀해주세요...
인식된 텍스트: 
OpenAI에게 질문 중...
OpenAI 응답: 안녕하세요! 궁금하신 점이나 도움이 필요하신 내용이 있으신가요? 무엇이든 질문해 주세요. 😊
음성으로 답변 중...
완료!
{
  "recognized_text": "",
  "ai_response": "안녕하세요! 궁금하신 점이나 도움이 필요하신 내용이 있으신가요? 무엇이든 질문해 주세요. 😊"
}
